# Stage 1 : Supervised Fine-tuning

## Setting up the model and dataset

In [ ]:
# pip install accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    device_map="auto",
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [2]:
from datasets import load_dataset

data = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
data

Dataset({
    features: ['output', 'instruction', 'input'],
    num_rows: 20022
})

In [ ]:
TOTAL_NUM = len(data)
TRAIN_SIZE = int(TOTAL_NUM * 0.4)
EVAL_SIZE = int(TRAIN_SIZE * 0.1)

train_data = data.select(range(TRAIN_SIZE - EVAL_SIZE))
eval_data = data.select(range(TRAIN_SIZE - EVAL_SIZE, TRAIN_SIZE))

In [ ]:
print("Size of train data : ", len(train_data))
print("Size of eval data : ", len(eval_data))

Size of train data : 7208
Size of eval data : 800


### Instruction Format for Fine-tuning

In [ ]:
def formatting_prompts_func(example):
    text = f"### Instruction: {example['instruction']}\n ### Output: {example['output']}"
    return text

## Fine-tuning

### SFT

In [ ]:
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

response_template = " ### Output:"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

train_config = SFTConfig(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=3,
    lr_scheduler_type="linear",
    logging_dir="logs",
    output_dir="full_ft",
    save_steps=500,
    logging_steps=50,
    bf16=True,
    save_total_limit=1,
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=train_config,
    formatting_func=formatting_prompts_func,
    data_collator=collator,
)

trainer.train()

Truncating train dataset:   0%|          | 0/7208 [00:00<?, ? examples/s]

[2025-03-29 21:07:47,129] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/shreyasjena/anaconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/shreyasjena/anaconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: jenashreyas to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
50,1.186000
100,1.146300
150,0.975900
200,1.005200
250,1.051100
300,0.969400
350,1.029000
400,0.919900
450,0.990800
500,0.898400


TrainOutput(global_step=2703, training_loss=0.4971754213218287, metrics={'train_runtime': 2125.8256, 'train_samples_per_second': 10.172, 'train_steps_per_second': 1.272, 'total_flos': 3.0206819147904e+16, 'train_loss': 0.4971754213218287})

### PEFT

In [7]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,
    target_modules=["q_proj", "v_proj"],
    task_type=TaskType.CAUSAL_LM,
    lora_alpha=8,
    lora_dropout=0.05,
)

In [9]:
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

response_template = " ### Output:"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

train_config = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    learning_rate=5e-5,
    num_train_epochs=3,
    lr_scheduler_type="linear",
    logging_dir="logs",
    output_dir="peft_ft",
    save_steps=500,
    logging_steps=50,
    bf16=True,
    save_total_limit=1,
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=train_config,
    peft_config=lora_config,        # passing this wraps the model in a PEFT wrapper 
    formatting_func=formatting_prompts_func,
    data_collator=collator,
)

trainer.train()

Applying formatting function to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/800 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

[2025-03-30 05:38:22,632] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/shreyasjena/anaconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/shreyasjena/anaconda3/envs/env/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for mor

Step,Training Loss
50,0.841200
100,0.604400
150,0.616500
200,0.603800
250,0.595600
300,0.599500
350,0.596900
400,0.591900
450,0.558500
500,0.557700


TrainOutput(global_step=1350, training_loss=0.5817779837714301, metrics={'train_runtime': 942.806, 'train_samples_per_second': 22.936, 'train_steps_per_second': 1.432, 'total_flos': 3.943450370440397e+16, 'train_loss': 0.5817779837714301})

## DARE : Model Merging using Mergekit

In [ ]:
!git clone https://github.com/arcee-ai/mergekit
%cd mergekit
%pip install -e .
%cd ..

Cloning into 'mergekit'...
remote: Enumerating objects: 2961, done.
remote: Counting objects: 100% (1224/1224), done.
remote: Compressing objects: 100% (379/379), done.
remote: Total 2961 (delta 1047), reused 861 (delta 845), pack-reused 1737 (from 3)
Receiving objects: 100% (2961/2961), 1.02 MiB | 2.49 MiB/s, done.
Resolving deltas: 100% (2025/2025), done.
/data2/shreyas/Safe-GenAI/Assignments/Assignment2/mergekit


/home/cds/.local/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


Now, we merge the `google/gemma-2-2b-it` base model with the fine-tuned SFT checkpoint (config defined in `sft_dare.yml`) using `mergekit` to obtain `gemma-2-2b-it-sft-dare`.

In [3]:
!cat configs/sft_dare.yml

base_model: google/gemma-2-2b-it
models:
  - model: jena-shreyas/gemma-2-2b-it-sft-code-alpaca
    parameters:
      weight: 1
      density: 0.3
merge_method: dare_linear
dtype: float16


In [ ]:
!mergekit-yaml configs/sft_dare.yml -v gemma-2-2b-it-sft-dare/

Warmup loader cache: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.29it/s]
INFO:mergekit.merge:Planning operations
INFO:root:Base model specified but not in input models - adding
INFO:root:Skipping optional weight lm_head.weight
Executing graph: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋| 1439/1442 [02:47<00:00,  5.71it/s]INFO:mergekit.io.tensor_writer:Writing shard #2 to disk
INFO:mergekit.io.tensor_writer:Finalizing shard names
Executing graph: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1442/1442 [02:48<00:00,  8.58it/s]
INFO:mergekit.merge:Saving config
Fetching 9 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 

Similarly, for `gemma-2-2b-it-peft-dare`, we first load the fine-tuned adapter checkpoint and merge with the base model:

In [1]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("google/gemma-2-2b-it")
peft_model_id = "jena-shreyas/gemma-2-2b-it-peft-code-alpaca"
model_ = PeftModel.from_pretrained(model, peft_model_id)
merged_model = model_.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

saving it as `gemma-2-2b-it-peft`...

In [ ]:
merged_model.save_pretrained("gemma-2-2b-it-peft")

Finally, we use mergekit to merge `gemma-2-2b-it-peft` with base model to create `gemma-2-2b-it-peft-dare`.

In [4]:
!cat configs/peft_dare.yml

base_model: google/gemma-2-2b-it
models:
  - model: gemma-2-2b-it-peft
    parameters:
      weight: 1
      density: 0.3
merge_method: dare_linear
dtype: float16


In [1]:
!mergekit-yaml configs/peft_dare.yml -v gemma-2-2b-it-peft-dare/

Warmup loader cache: 100%|████████████████████████| 2/2 [00:01<00:00,  1.54it/s]
INFO:mergekit.merge:Planning operations
INFO:root:Base model specified but not in input models - adding
INFO:root:Skipping optional weight lm_head.weight
Executing graph: 100%|█████████████████████▉| 1435/1442 [00:51<00:00, 35.97it/s]INFO:mergekit.io.tensor_writer:Writing shard #2 to disk
INFO:mergekit.io.tensor_writer:Finalizing shard names
Executing graph: 100%|██████████████████████| 1442/1442 [00:51<00:00, 27.99it/s]
INFO:mergekit.merge:Saving config
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 20821.15it/s]
INFO:mergekit.merge:Copying tokenizer from google/gemma-2-2b-it
Fetching 9 files: 100%|████████████████████████| 9/9 [00:00<00:00, 63872.65it/s]
